# Gene expression analyses
## get the gene expression of the Allen human brain atlas, then explore the correlation between the gene expression and the LNM or other maps
## Xiao Chen 260517
## chenxiaophd@gmail.com

In [1]:
import os
import sys
#import abagen
import pandas as pd
from pathlib import Path
import numpy as np
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
import nibabel as nib
#import gseapy as gp
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

%load_ext autoreload
%autoreload 2
sys.path.append("../scripts/functions")

#from AHBA_preprocessing import *

## read in the LNM data and extract pacellation data

In [2]:
work_dir = "xxxxxxxxxxx"
# the AHBA only has the left hemisphere
LNM_path = os.path.join(work_dir, "/rumination_MDD_GSR_lh.gii")
atlas_path = "/dpabi_dir/DPABISurf/SurfTemplates/fsaverage5_lh_Schaefer2018_400Parcels_7Networks_order.label.gii"
out_csv = os.path.join(work_dir, "gene_expression/rum_MDD_lnm_vec_left_schaefer400.csv")

lnm = nib.load(LNM_path).darrays[0].data
atlas = nib.load(atlas_path).darrays[0].data.astype(int)

# check dimensions
assert len(lnm) == len(atlas)

# left hemisphere parcels
parcel_ids = np.arange(1,201)
lnm_mean = np.full(200, np.nan)

for i, roi in enumerate(parcel_ids):
    mask = atlas == roi
    if np.any(mask):
        lnm_mean[i] = np.nanmean(lnm[mask])

# 200×1 vector
lnm_vec = lnm_mean.reshape(-1,1)

# save as csv
pd.DataFrame(
    lnm_vec,
    columns=["LNM"]
).to_csv(
    out_csv,
    index=False
)

print(f"Saved to:\n{out_csv}")


Saved to:
/Users/Chenxiao/Documents/project_codes/rumination_casual_mapping/analyses/stable_260725/gene_expression/rum_MDD_lnm_vec_left_schaefer400.csv


## download the abagen files
## already downloaded, no need to download again

In [ ]:
# files = abagen.fetch_microarray(donors='all', verbose=0)

## do preprocessing, get the gene expression data

In [ ]:
# Replace original function in abagen package with patched function ###
# omit the brain regions that are far from cortex
abagen.samples_.drop_mismatch_samples = drop_mismatch_samples_and_filter
atlas = fetch_schaefer(size=400)

In [ ]:
expression, counts, report = get_expression_abagen(
    atlas=atlas,
    data_dir="/xxxxxxx/abagen-data",
    save_name="AHBA_schaefer400_left_cortex_stable_260529_v2",

    # to explore, no threshold
    donors_threshold=3,
    gene_stability_threshold=0.5,

    # QC info
    return_counts=True,
    return_report=True,
    lr_mirror="rightleft"        

    # # abagen preprocessing parameters
    # ibf_threshold=0.5,              # intensity-based filter: 50%
    # probe_selection="diff_stability",
    # tolerance=2,                    # 2 mm voxel-mass matching
    # sample_norm="srs",              # scaled robust sigmoid
    # gene_norm="srs",
    # region_agg="donors",
    # agg_metric="mean",
    # verbose=1
)